<a href="https://colab.research.google.com/github/Adhira-Deogade/pytorch-learnings/blob/main/notebooks/trasnfer_learning_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Reference: https://github.com/rasbt/stat453-deep-learning-ss21/blob/main/L14/5-transfer-learning-vgg16_small.ipynb

In [4]:
from torch.utils.data import sampler
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision import datasets, transforms

In [5]:
# Normalized Value = (Original Value - Mean) / Standard Deviation
# Let's create an UnNormalizer
class UnNormalizer(object):
  def __init__(self, mean, std):
    self.mean = mean
    self.std = std

  def __clall__(self, tensor):
    for tsor, m, s in zip(tensor, self.mean, self.std):
      tsor.mul_(s).add_(m)
    return tensor


In [6]:
RANDOM_SEED = 900
BATCH_SIZE = 256
NUM_EPOCHS = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device = {device}")

Device = cuda


In [7]:
# train and test transforms as defined by creators of Pre-trained NN
# Resize image to 70, 70
# Get a random crop of size 64, 64
# Convert the image to tensor
# Normalize the tensor as defined by mean and standard deviation
train_transform = torchvision.transforms.Compose([
    transforms.Resize((70, 70)),
    transforms.RandomCrop((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = torchvision.transforms.Compose([
    transforms.Resize((70, 70)),
    transforms.RandomCrop((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])


In [9]:
def get_dataloader_cifar100(batch_size, num_workers, validation_fraction, train_transforms, test_transforms):
  if train_transforms is None:
    train_transforms = torchvision.transforms.ToTensor()
  if test_transforms is None:
    test_transforms = torchvision.transforms.ToTensor()

  train_dataset  = datasets.CIFAR100(train=True,
                                     transform=train_transforms,
                                     download=True,
                                     root='data')

  test_dataset = datasets.CIFAR100(train=False,
                                   transform=test_transforms,
                                   root='data')
  # Index sampler for train and validation split
  if validation_fraction is not None:
    train_dataset_size = len(train_dataset)
    train_validation_split_index = int(validation_fraction * train_dataset_size)
    print(f"train_validation_split_index = {train_validation_split_index}")

    train_indices = torch.arange(0, train_dataset_size - train_validation_split_index)
    validation_indices = torch.arange(train_dataset_size - train_validation_split_index, train_dataset_size)

    train_sampler = SubsetRandomSampler(train_indices)
    validation_sampler = SubsetRandomSampler(validation_indices)

    valid_dataset = datasets.CIFAR100(train=True,
                                    transform=test_transforms,
                                    root='data')
    train_loader = DataLoader(train_dataset,
                              batch_size=batch_size,
                              num_workers=num_workers,
                              sampler=train_sampler,
                              drop_last=True)

    validation_loader = DataLoader(valid_dataset,
                                   batch_size=batch_size,
                                   num_workers=num_workers,
                                   sampler=validation_sampler)
  else:
    train_loader = DataLoader(train_dataset,
                              batch_size=batch_size,
                              num_workers=num_workers,
                              drop_last=True,
                              shuffle=True)
  print(f"Train loader length = {len(train_loader)}")
  test_loader = DataLoader(test_dataset,
                          batch_size=batch_size,
                          num_workers=num_workers,
                          drop_last=True,
                           shuffle=True)
  if validation_fraction is not None:
    return train_loader, validation_loader, test_loader
  else:
    return train_loader, test_loader



In [10]:
train_loader, valid_loader, test_loader = get_dataloader_cifar100(batch_size=BATCH_SIZE,
                                                                  validation_fraction=0.1,
                                                                  train_transforms=train_transform,
                                                                  test_transforms=test_transform,
                                                                  num_workers=2)

100%|██████████| 169M/169M [00:05<00:00, 29.4MB/s]


train_validation_split_index = 5000
Train loader length = 175


In [11]:
for image, label in train_loader:
  print(f"Image batch size - {image.shape}")
  print(f"Label batch size - {label.shape}")
  print(f"Actual labels - {label[:10]}")
  break

Image batch size - torch.Size([256, 3, 64, 64])
Label batch size - torch.Size([256])
Actual labels - tensor([41, 56, 83, 72, 46, 78, 58, 94, 98, 53])


In [12]:
# Load a pre-trained model
model = torchvision.models.resnet101(pretrained=True)
print(model)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth
100%|██████████| 171M/171M [00:01<00:00, 174MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [13]:
print(model.fc)

Linear(in_features=2048, out_features=1000, bias=True)


In [14]:
# Freeze all layers
for param in model.parameters():
  param.requires_grad = False


In [15]:
# Unfreeze the last layer4 and fully connect layer


In [16]:
# # Replace last fully connected layer with a new layer
# # The new layer has same size as the size of new dataset classes - here 100 (Cifar100)


In [17]:
print(model.layer4[2])

Bottleneck(
  (conv1): Conv2d(2048, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(512, 2048, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn3): BatchNorm2d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
)


In [18]:
for param in (model.layer4[2].parameters()):
  param.requires_grad = True

In [19]:
print(model.fc)

Linear(in_features=2048, out_features=1000, bias=True)


In [20]:
model.fc = torch.nn.Linear(2048, 100)

In [21]:
# Train the model as usual

In [22]:
for batch_idx, (features, targets) in enumerate(train_loader):
  print(batch_idx)
  print(features.shape)
  print(targets.shape)
  break

0
torch.Size([256, 3, 64, 64])
torch.Size([256])


In [23]:
for features, targets in train_loader:
  # print(batch_idx)
  print(features.shape)
  print(targets.shape)
  print(targets.size()[0])
  break

torch.Size([256, 3, 64, 64])
torch.Size([256])
256


In [24]:
def compute_accuracy(model, data_loader, device):
  """
  Return accuracy in % for given data_loader
  """
  with torch.no_grad():
    correct_predictions, total_examples = 0, 0

    for features, labels in data_loader:
      features = features.to(device)
      labels = labels.to(device)

      logits = model(features)
      # Returns value, indices
      _, predicted_labels = torch.max(logits, 1) # 1 means along the row

      total_examples += labels.size(0) # This is a single vector, to get single value
      correct_predictions += (predicted_labels == labels).sum()

    accuracy = 100 * correct_predictions / total_examples
  return accuracy


In [25]:
import time

In [26]:
def train_model(
    model,
    num_epochs,
    train_loader,
    validation_loader,
    test_loader,
    optimizer,
    device,
    scheduler=None,
    scheduler_on='valid_acc',
    logging_interval=50):
  start_time = time.time()
  minibatch_train_acc, train_acc_list, val_acc_list = [], [], []

  for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, targets) in enumerate(train_loader):
      features = features.to(device)
      targets = targets.to(device)

      logits = model(features)
      loss = torch.nn.functional.cross_entropy(logits, targets)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      # Logging
      minibatch_train_acc.append(loss.item())
      # For every 50 batches, print log message
      if (batch_idx+1)%logging_interval == 0:
        print(f"Epoch = {epoch+1}/{num_epochs}; Batch = {batch_idx+1}/{len(train_loader)}; Loss = {loss}")

    # Let's get validation loss
    model.eval()
    with torch.no_grad():
      train_acc = compute_accuracy(model, train_loader, device)
      val_acc = compute_accuracy(model, validation_loader, device)
      train_acc_list.append(train_acc.item())
      val_acc_list.append(val_acc.item())
      print(f"Epoch = {epoch+1}/{num_epochs}; Train Accuracy = {train_acc}; Validation Accuracy = {val_acc}")

    # Total time elapsed during each epoch
    time_elapsed = time.time() - start_time
    print(f"Epoch time elapsed = {time_elapsed}")
    if scheduler is not None:
      if scheduler_on == 'valid_acc':
        scheduler.step(val_acc_list[-1])
      elif scheduler_on == 'minibatch_acc':
        scheduler.step(train_acc_list[-1])
      else:
        raise ValueError(f"Invalid {scheduler_on} choice")

  # Total training time elapsed
  total_time_elapsed = (time.time() - start_time)/60
  print(f"Total training time elapsed = {total_time_elapsed} min")

  # Test accuracy
  model.eval()
  with torch.no_grad():
    test_acc = compute_accuracy(model, test_loader, device)
    print(f"Test Accuracy = {test_acc}")

  return minibatch_train_acc, train_acc_list, val_acc_list

In [27]:
# Training the model
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, verbose=True)

minibatch_acc_list, train_acc_list, validate_acc_list = train_model(model=model,
                                                                  num_epochs=NUM_EPOCHS,
                                                                  train_loader=train_loader,
                                                                  validation_loader=valid_loader,
                                                                    device=device,
                                                                  test_loader=test_loader,
                                                                    optimizer=optimizer,
                                                                    scheduler=scheduler,
                                                                    scheduler_on='valid_acc')

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch = 1/50; Batch = 50/175; Loss = 2.459057331085205
Epoch = 1/50; Batch = 100/175; Loss = 2.13490629196167
Epoch = 1/50; Batch = 150/175; Loss = 2.054269313812256
Epoch = 1/50; Train Accuracy = 53.72991180419922; Validation Accuracy = 48.68000030517578
Epoch time elapsed = 55.73166537284851
Epoch = 2/50; Batch = 50/175; Loss = 1.6082361936569214
Epoch = 2/50; Batch = 100/175; Loss = 1.5830594301223755
Epoch = 2/50; Batch = 150/175; Loss = 1.7015424966812134
Epoch = 2/50; Train Accuracy = 59.55133819580078; Validation Accuracy = 52.15999984741211
Epoch time elapsed = 109.66512560844421
Epoch = 3/50; Batch = 50/175; Loss = 1.4758155345916748
Epoch = 3/50; Batch = 100/175; Loss = 1.4458931684494019
Epoch = 3/50; Batch = 150/175; Loss = 1.5752530097961426
Epoch = 3/50; Train Accuracy = 64.70982360839844; Validation Accuracy = 53.18000030517578
Epoch time elapsed = 165.13612723350525
Epoch = 4/50; Batch = 50/175; Loss = 1.1281423568725586
Epoch = 4/50; Batch = 100/175; Loss = 1.208117246

In [28]:
print('hello')

hello
